[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/03_tensor_pipeline_parallel/03_tensor_pipeline_parallel.ipynb)

# 03 · 张量与流水并行（用 numpy 模拟多 rank）

**目标**：用 numpy 在单进程内**模拟多 rank**，把 Megatron 张量并行（列切→行切→一次 all-reduce）和流水线并行（micro-batch、bubble、1F1B）的核心机制从零实现，并与**单卡参考对拍到 1e-10**。

**路线**：TP 列切分 → TP 行切分+all-reduce → 完整 MLP 块（数通信次数）→ bubble 公式 → 朴素 vs 1F1B 调度模拟 → 激活显存峰值 → ✏️ 4 道练习 → 📖 答案 → 🧪 真实模型（GPT-3）胶囊 → 🔧 Megatron 真实代码对照。

> 心智模型：**一个 rank = 一份切片 + 一段循环；all-reduce = 对 rank 列表求和；bubble = 流水线调度表里的空闲时隙**。我们写的是并行**结构与正确性**，不是真实多卡性能。

## 1 · 张量并行：列切分（column parallel）

FFN 第一层 `Y = GeLU(X @ A)`，`A` 是 `[h, 4h]`。把 `A` 沿**列**切成 `t` 块 `A=[A_0|A_1|...]`，rank `i` 持有完整的 `X` 与自己的 `A_i`，独立算 `Y_i = GeLU(X @ A_i)`（**零通信**），输出按列拼接。

**对拍**：拼接后的 `[Y_0|Y_1|...]` 必须逐位等于单卡 `GeLU(X @ A)`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def gelu(x):
    # tanh 近似版 GeLU（逐元素，TP 列切分依赖它逐元素这一性质）
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def column_parallel_linear(X, A, t):
    '''把 A 沿列(轴1)切成 t 份，模拟 t 个 rank 各算一块 GeLU(X@A_i)，返回每个 rank 的输出列表。'''
    A_shards = np.split(A, t, axis=1)          # 沿输出维切：每块 [h, 4h/t]
    Y_shards = [gelu(X @ A_i) for A_i in A_shards]  # 每个 rank 独立算，无通信
    return Y_shards

s, h, t = 6, 8, 4                              # 序列×batch=6, hidden=8, TP 度=4
X = rng.standard_normal((s, h))
A = rng.standard_normal((h, 4*h))

Y_ref = gelu(X @ A)                            # 单卡参考
Y_shards = column_parallel_linear(X, A, t)
Y_tp = np.concatenate(Y_shards, axis=1)        # 各 rank 输出按列拼回

print('单卡 Y 形状      :', Y_ref.shape)
print('每个 rank 输出形状:', Y_shards[0].shape, f'(共 {t} 个 rank)')
print('列切分前向通信次数:', 0, '(GeLU 逐元素，无需通信)')
assert np.allclose(Y_tp, Y_ref, atol=1e-10), '列切分拼接应逐位等于单卡'
print('✅ 对拍通过：列切分 + 拼接 == 单卡 GeLU(X@A)，且零通信')

## 2 · 张量并行：行切分 + all-reduce

FFN 第二层 `Z = Y @ B`，`B` 是 `[4h, h]`。上一节每个 rank 手里是 `Y_i`（`[s, 4h/t]`）。把 `B` 沿**行**切成 `B=[B_0; B_1; ...]`（每块 `[4h/t, h]`），rank `i` 算**部分和** `Z_i = Y_i @ B_i`，真正结果是所有部分和之**和** —— 这就是 **all-reduce**！

**对拍**：`sum_i (Y_i @ B_i)` 必须等于单卡 `Y @ B`。

In [ ]:
def all_reduce_sum(tensor_list):
    '''模块 01 的 all-reduce（求和版）：把各 rank 的张量相加，结果每个 rank 一致。'''
    total = np.zeros_like(tensor_list[0])
    for t_i in tensor_list:
        total = total + t_i                    # 真实硬件用 ring all-reduce，这里直接求和对拍
    return [total.copy() for _ in tensor_list]  # 每个 rank 拿到相同的总和

def row_parallel_linear(Y_shards, B, t):
    '''B 沿行(轴0)切 t 份；rank i 用自己的 Y_i 和 B_i 算部分和，再 all-reduce 求和。'''
    B_shards = np.split(B, t, axis=0)          # 沿输入维切：每块 [4h/t, h]
    partial = [Y_shards[i] @ B_shards[i] for i in range(t)]  # 每 rank 一个部分和 [s, h]
    Z_each = all_reduce_sum(partial)          # 1 次 all-reduce 把部分和加起来
    return Z_each, len(partial)

B = rng.standard_normal((4*h, h))
Y = np.concatenate(Y_shards, axis=1)          # 完整 Y（仅用于参考，TP 中并不真的拼）
Z_ref = Y @ B                                  # 单卡参考

Z_each, n_partial = row_parallel_linear(Y_shards, B, t)
print('部分和个数(=rank 数):', n_partial)
print('行切分前向通信次数  :', 1, '(一次 all-reduce 求和)')
# 每个 rank 都应拿到相同且正确的结果
for i in range(t):
    assert np.allclose(Z_each[i], Z_ref, atol=1e-10), f'rank {i} 结果应等于单卡 Y@B'
assert all(np.allclose(Z_each[0], z) for z in Z_each), 'all-reduce 后所有 rank 须一致'
print('✅ 对拍通过：sum_i(Y_i @ B_i) == 单卡 Y@B，且所有 rank 一致')

## 3 · 完整 FFN 块：列切→行切，整块只通信一次

把前两节拼起来：`Z = GeLU(X @ A) @ B`。Megatron 的设计让中间的 `Y_i` **保持分片、不拼回**，一路算到第二层的部分和，**整个 FFN 块前向只需 1 次 all-reduce**。

**对拍**：TP 版完整块 == 单卡完整块；并确认通信次数 = 1。

In [ ]:
def mlp_block_single(X, A, B):
    return gelu(X @ A) @ B                     # 单卡参考：一整块

def mlp_block_tp(X, A, B, t):
    '''Megatron FFN 块：列切(第一层,0通信) -> 中间保持分片 -> 行切(第二层) -> 1次 all-reduce。'''
    comm = 0
    # 第一层：列切分，每 rank 算 Y_i = GeLU(X @ A_i)，零通信
    A_shards = np.split(A, t, axis=1)
    Y_shards = [gelu(X @ A_i) for A_i in A_shards]
    # 第二层：行切分，每 rank 算部分和 Z_i = Y_i @ B_i
    B_shards = np.split(B, t, axis=0)
    partial = [Y_shards[i] @ B_shards[i] for i in range(t)]
    # 唯一一次通信：all-reduce 求和
    Z_each = all_reduce_sum(partial); comm += 1
    return Z_each[0], comm

Z_single = mlp_block_single(X, A, B)
Z_tp, comm = mlp_block_tp(X, A, B, t)
print(f'单卡 FFN 块输出形状 : {Z_single.shape}')
print(f'TP({t}) FFN 块前向通信 : {comm} 次 all-reduce')
assert np.allclose(Z_tp, Z_single, atol=1e-10), 'TP FFN 块应逐位等于单卡'
assert comm == 1, '一个 FFN 块前向只应有 1 次 all-reduce'
print('✅ 对拍通过：整个 FFN 块用 TP 切分后 == 单卡，且只通信 1 次（这是 Megatron 的核心设计）')

## 4 · 流水线 bubble 公式

PP 把 `L` 层切成 `p` 个 stage。把 batch 切成 `m` 个 micro-batch 填充流水线后，bubble（空闲）占比：

$$\text{bubble fraction} = \frac{p-1}{m + p - 1}$$

把它从零实现，并与手算值对拍；再打一张表看 micro-batch 如何稀释 bubble。

In [ ]:
def bubble_fraction(p, m):
    '''p 个 stage、m 个 micro-batch 的流水线气泡占比。'''
    assert p >= 1 and m >= 1
    return (p - 1) / (m + p - 1)

# 与手算对拍
assert abs(bubble_fraction(4, 1) - 3/4) < 1e-12     # 朴素：利用率仅 25%
assert abs(bubble_fraction(4, 8) - 3/11) < 1e-12
assert abs(bubble_fraction(1, 5) - 0.0) < 1e-12     # 单 stage 无 bubble

print(f"{'p (stage)':>10}{'m (micro)':>12}{'bubble':>10}{'利用率':>10}")
for p, m in [(4,1),(4,4),(4,8),(4,16),(4,32),(8,32)]:
    b = bubble_fraction(p, m)
    print(f'{p:>10}{m:>12}{b:>9.1%}{1-b:>10.1%}')
print('\n✅ bubble 公式验证通过：m 越大 bubble 越小；经验法则 m>=4p 把 bubble 压到 ~10%')

## 5 · 调度模拟：朴素 vs 1F1B 的时隙账

用 numpy 模拟流水线**调度表**：行=stage，列=时间步，记录每个 (stage, t) 在忙还是空闲（bubble）。

朴素 GPipe（全前向再全反向）和 1F1B 的 **bubble 占比相同**（都由公式决定），下面通过统计调度表里的空闲格子来**验证**这一点。

In [ ]:
def schedule_total_slots(p, m):
    '''一次前向+反向的总时间步：F 阶段 (m+p-1) + B 阶段 (m+p-1)。'''
    return 2 * (m + p - 1)

def busy_slots(p, m):
    '''真正干活的格子数：每个 micro-batch 在每个 stage 各做一次 F 和一次 B。'''
    return 2 * p * m

def bubble_from_slots(p, m):
    '''用「空闲格子 / 总格子」反推 bubble，应与公式一致。'''
    total_cells = p * schedule_total_slots(p, m)   # p 行 × 总时间步
    busy = busy_slots(p, m)
    idle = total_cells - busy
    return idle / total_cells

p, m = 4, 8
print(f'p={p}, m={m}:')
print(f'  总时间步        = {schedule_total_slots(p, m)}')
print(f'  忙碌格子        = {busy_slots(p, m)}')
print(f'  空闲格子(bubble)= {p*schedule_total_slots(p,m) - busy_slots(p,m)}')
frac_slots = bubble_from_slots(p, m)
frac_formula = bubble_fraction(p, m)
print(f'  bubble(数格子)  = {frac_slots:.4f}')
print(f'  bubble(公式)    = {frac_formula:.4f}')
assert abs(frac_slots - frac_formula) < 1e-12, '数格子与公式应一致'
print('✅ 朴素与 1F1B 的 bubble 占比相同（都由 (p-1)/(m+p-1) 决定）—— 1F1B 省的是显存不是 bubble')

## 6 · 激活显存峰值：GPipe vs 1F1B

**这才是 1F1B 的真正价值**。GPipe 先做完所有 `m` 个前向再统一反向 → 必须同时存 `m` 份激活。1F1B 让每个 micro-batch 的反向尽早执行、尽早释放激活 → 峰值降到约 `p` 份。

模拟一个 stage 的「在飞激活数」随调度推进的变化，取峰值对比。

In [ ]:
def peak_activations_gpipe(p, m):
    '''GPipe：F F F ... B B B。前向阶段激活单调累积到 m 份才开始释放。'''
    live = 0; peak = 0
    for _ in range(m):       # m 次前向，每次 +1 份激活
        live += 1; peak = max(peak, live)
    for _ in range(m):       # m 次反向，每次 -1 份
        live -= 1
    return peak

def peak_activations_1f1b(p, m):
    '''1F1B：填充 (p-1) 个前向后进入 F/B 交替，在飞激活稳定在 p 份(峰值)。'''
    live = 0; peak = 0
    warmup = min(p - 1, m)   # 填充期：做 p-1 个前向（第 p 个前向后紧跟第 1 个反向）
    for _ in range(warmup):
        live += 1; peak = max(peak, live)
    for _ in range(m - warmup):   # 稳态：一前向(+1)立刻一反向(-1)，峰值出现在前向后
        live += 1; peak = max(peak, live)
        live -= 1
    return peak

p, m = 4, 16
g = peak_activations_gpipe(p, m)
f = peak_activations_1f1b(p, m)
print(f'p={p}, m={m}:  GPipe 激活峰值 = {g} 份   1F1B 激活峰值 = {f} 份')
print(f'1F1B 省显存约 {g/f:.1f}x')
assert g == m, 'GPipe 峰值应为 m 份'
assert f <= p, '1F1B 峰值应 <= p 份'
assert f < g, '1F1B 峰值必须低于 GPipe'
print('✅ 对拍通过：bubble 一样，但 1F1B 激活峰值从 m 份降到 ~p 份 —— 这才是它能撑更深流水线的原因')

---
## ✏️ 练习 1：自己实现一个 TP 线性层（列切 + 行切）

实现 `tp_two_layer(X, A, B, t)`：把 `Z = (X @ A) @ B`（这里**不加** GeLU，纯线性，便于验证）用「`A` 列切 → `B` 行切 → all-reduce」做出来，返回 `(Z, 通信次数)`。

提示：纯线性时 `(X @ A) @ B = X @ (A @ B)`，但**不要**这样抄近路——要真正按 TP 切分算，才能验证切分逻辑。

In [ ]:
def tp_two_layer(X, A, B, t):
    comm = 0
    # TODO:
    #  1) A 沿列(axis=1)切 t 份，rank i 算 Y_i = X @ A_i（无 GeLU，零通信）
    #  2) B 沿行(axis=0)切 t 份，rank i 算部分和 Z_i = Y_i @ B_i
    #  3) all_reduce_sum 求和，comm += 1
    #  返回 (Z, comm)
    raise NotImplementedError
    return Z, comm

In [ ]:
# —— 练习 1 自测 ——
s, h, t = 5, 8, 4
X = rng.standard_normal((s, h)); A = rng.standard_normal((h, 4*h)); B = rng.standard_normal((4*h, h))
Z, comm = tp_two_layer(X, A, B, t)
Z_ref = (X @ A) @ B
assert Z.shape == (s, h)
assert np.allclose(Z, Z_ref, atol=1e-10), 'TP 两层应等于单卡 (X@A)@B'
assert comm == 1, '只应有 1 次 all-reduce'
print('✅ 练习 1 通过：列切+行切+一次 all-reduce == 单卡，通信次数=1')

## ✏️ 练习 2：注意力按头切分（head parallel）

多头注意力本就是若干独立的头并排。实现 `attention_tp(X, Wq, Wk, Wv, Wo, n_heads, t)`：
把 `n_heads` 个头**均分**给 `t` 个 rank，每 rank 算自己头的注意力，输出投影 `Wo` 按**行**切，最后 all-reduce。
（用最简单的单 query 全注意力：`softmax(Q K^T / sqrt(d)) V`，无 mask。）返回 `(out, 通信次数)`。

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def attention_single(X, Wq, Wk, Wv, Wo, n_heads):
    '''单卡多头注意力参考实现。X:[s,h]  W*:[h,h]  返回 [s,h]。'''
    s, h = X.shape; d = h // n_heads
    Q, Kk, V = X @ Wq, X @ Wk, X @ Wv          # [s,h]
    outs = []
    for hd in range(n_heads):
        sl = slice(hd*d, (hd+1)*d)
        q, k, v = Q[:, sl], Kk[:, sl], V[:, sl]
        a = softmax(q @ k.T / np.sqrt(d))
        outs.append(a @ v)                      # [s,d]
    ctx = np.concatenate(outs, axis=1)          # [s,h]
    return ctx @ Wo

def attention_tp(X, Wq, Wk, Wv, Wo, n_heads, t):
    comm = 0
    # TODO:
    #  - 每个 rank 负责 n_heads/t 个头：算这些头的 Q/K/V 与注意力，得到该 rank 的 ctx 片 [s, h/t]
    #  - Wo 沿行(axis=0)切 t 份，rank i 算部分和 ctx_i @ Wo_i  -> [s,h]
    #  - all_reduce_sum 求和，comm += 1；返回 (out, comm)
    raise NotImplementedError
    return out, comm

In [ ]:
# —— 练习 2 自测 ——
s, h, n_heads, t = 4, 8, 4, 2                  # 4 个头分给 2 个 rank，每 rank 2 个头
X = rng.standard_normal((s, h))
Wq, Wk, Wv, Wo = (rng.standard_normal((h, h)) for _ in range(4))
out, comm = attention_tp(X, Wq, Wk, Wv, Wo, n_heads, t)
ref = attention_single(X, Wq, Wk, Wv, Wo, n_heads)
assert out.shape == (s, h)
assert np.allclose(out, ref, atol=1e-10), '按头切的注意力应等于单卡'
assert comm == 1, '注意力 TP 前向应只有 1 次 all-reduce'
print('✅ 练习 2 通过：注意力按头切分 + Wo 行切 + 一次 all-reduce == 单卡')

## ✏️ 练习 3：bubble 调参 —— 求满足目标的最小 micro-batch

给定 stage 数 `p` 和目标 bubble 上限 `target`（如 0.05），实现 `min_microbatch(p, target)`：
返回使 `bubble_fraction(p, m) <= target` 的**最小** `m`。

提示：解 `(p-1)/(m+p-1) <= target` 得 `m >= (p-1)/target - (p-1) = (p-1)(1-target)/target`，向上取整。

In [ ]:
import math
def min_microbatch(p, target):
    # TODO: 返回满足 bubble_fraction(p,m) <= target 的最小整数 m（m>=1）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for p, target in [(4, 0.10), (8, 0.05), (16, 0.05), (2, 0.20)]:
    m = min_microbatch(p, target)
    assert m >= 1 and isinstance(m, int)
    assert bubble_fraction(p, m) <= target + 1e-12, f'm={m} 未达标'
    assert bubble_fraction(p, m-1) > target - 1e-12 if m > 1 else True, 'm 应是最小值'
    print(f'p={p:2d}, 目标 bubble<={target:.0%} -> 最小 micro-batch m={m} (实际 bubble={bubble_fraction(p,m):.1%})')
print('✅ 练习 3 通过：能据目标 bubble 算出最小 micro-batch（注意 p 越大需要的 m 越多）')

## ✏️ 练习 4：3D 并行的 rank ↔ 坐标映射

3D 并行里每个 rank 对应一个 `(dp, pp, tp)` 坐标。约定 **TP 维变化最快**（相邻 rank 在同一节点内），即 `rank = dp*(pp_size*tp_size) + pp*tp_size + tp`。

实现 `rank_to_coord(rank, dp_size, pp_size, tp_size)` 和 `coord_to_rank(dp,pp,tp, ...)`，并验证：(1) 两者互逆（往返一致）；(2) 遍历 `0..world_size-1` 恰好覆盖所有坐标一次。

In [ ]:
def coord_to_rank(dp, pp, tp, dp_size, pp_size, tp_size):
    # TODO: 按 rank = dp*(pp_size*tp_size) + pp*tp_size + tp 返回 rank
    raise NotImplementedError

def rank_to_coord(rank, dp_size, pp_size, tp_size):
    # TODO: 逆运算，返回 (dp, pp, tp)
    #   tp = rank % tp_size; pp = (rank // tp_size) % pp_size; dp = rank // (tp_size*pp_size)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
dp_size, pp_size, tp_size = 2, 3, 4
world = dp_size * pp_size * tp_size            # = 24
seen = set()
for r in range(world):
    dp, pp, tp = rank_to_coord(r, dp_size, pp_size, tp_size)
    assert 0 <= dp < dp_size and 0 <= pp < pp_size and 0 <= tp < tp_size
    assert coord_to_rank(dp, pp, tp, dp_size, pp_size, tp_size) == r, '往返必须一致'
    seen.add((dp, pp, tp))
assert len(seen) == world, f'应覆盖全部 {world} 个坐标，实际 {len(seen)}'
print(f'world_size = {dp_size}×{pp_size}×{tp_size} = {world}')
print(f'rank 0  -> {rank_to_coord(0, dp_size, pp_size, tp_size)}')
print(f'rank 5  -> {rank_to_coord(5, dp_size, pp_size, tp_size)}  (TP 维变化最快)')
print(f'rank 23 -> {rank_to_coord(23, dp_size, pp_size, tp_size)}')
print('✅ 练习 4 通过：rank↔(dp,pp,tp) 互逆且双射 —— 这是建通信组的基础')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def tp_two_layer(X, A, B, t):
    comm = 0
    A_shards = np.split(A, t, axis=1)
    Y_shards = [X @ A_i for A_i in A_shards]            # 列切，零通信
    B_shards = np.split(B, t, axis=0)
    partial = [Y_shards[i] @ B_shards[i] for i in range(t)]
    Z = all_reduce_sum(partial)[0]; comm += 1          # 一次 all-reduce
    return Z, comm

In [ ]:
# 练习 2 参考答案
def attention_tp(X, Wq, Wk, Wv, Wo, n_heads, t):
    comm = 0
    s, h = X.shape; d = h // n_heads; heads_per_rank = n_heads // t
    Wo_shards = np.split(Wo, t, axis=0)                 # Wo 按行切 [h/t, h]
    partial = []
    for i in range(t):
        ctx_cols = []
        for hd in range(i*heads_per_rank, (i+1)*heads_per_rank):
            sl = slice(hd*d, (hd+1)*d)
            q = X @ Wq[:, sl]; k = X @ Wk[:, sl]; v = X @ Wv[:, sl]
            a = softmax(q @ k.T / np.sqrt(d))
            ctx_cols.append(a @ v)                      # [s,d]
        ctx_i = np.concatenate(ctx_cols, axis=1)        # [s, h/t]
        partial.append(ctx_i @ Wo_shards[i])           # 部分和 [s,h]
    out = all_reduce_sum(partial)[0]; comm += 1
    return out, comm

In [ ]:
# 练习 3 参考答案
def min_microbatch(p, target):
    if p == 1:
        return 1                                        # 单 stage 无 bubble
    m = math.ceil((p - 1) * (1 - target) / target)
    return max(1, m)

In [ ]:
# 练习 4 参考答案
def coord_to_rank(dp, pp, tp, dp_size, pp_size, tp_size):
    return dp * (pp_size * tp_size) + pp * tp_size + tp

def rank_to_coord(rank, dp_size, pp_size, tp_size):
    tp = rank % tp_size
    pp = (rank // tp_size) % pp_size
    dp = rank // (tp_size * pp_size)
    return dp, pp, tp

---
## 🧪 真实数据胶囊：给 GPT-3 175B 配 3D 并行

用**真实** GPT-3 175B 的配置（`hidden=12288, layers=96, heads=96`，来自 Brown 2020），算一套合理的 3D 并行方案：单层 FFN 参数量、TP 切分后每卡份额、PP 切分后每卡层数、bubble。

（纯算术，`try/except` 仅为风格统一；数字全部内置，不联网。）

In [ ]:
def gpt3_config():
    '''GPT-3 175B 公开配置（Brown et al. 2020, Table 2.1）。'''
    try:
        # 真实训练里可能从 config.json 读；本环境直接用公开数字
        raise RuntimeError('offline')
    except Exception:
        return dict(hidden=12288, layers=96, heads=96, ffn_mult=4)

cfg = gpt3_config()
h, L, H = cfg['hidden'], cfg['layers'], cfg['heads']
# 单层 FFN 两个矩阵的参数量 [h,4h]+[4h,h] = 8 h^2
ffn_params = 8 * h**2
print(f"GPT-3 175B: hidden={h}, layers={L}, heads={H}")
print(f'单层 FFN 参数量 = 8*h^2 = {ffn_params/1e6:.0f}M (单卡 fp16 仅权重就 {ffn_params*2/1e9:.2f} GB)')

# 一套常见方案：tp=8(节点内), pp=12, 余下做 dp
tp, pp = 8, 12
assert H % tp == 0, 'TP 度须整除头数'
assert L % pp == 0, 'PP 度须整除层数'
print(f'\nTP={tp}: 每卡 FFN 份额 = {ffn_params/tp/1e6:.0f}M 参数, {H//tp} 个注意力头')
print(f'PP={pp}: 每个 stage = {L//pp} 层')
m = 4 * pp                                     # 经验：m>=4p
print(f'micro-batch m={m}: bubble = {bubble_fraction(pp, m):.1%}')
print(f'若要 {tp}×{pp}=96 卡基础上再 dp=8 扩 batch -> world_size = {tp*pp*8} 卡')

**🧪 胶囊练习**：实现 `cards_needed(total_params_b, per_card_gb, bytes_per_param=2)`：
给定模型总参数（十亿，fp16）和单卡可用显存（GB），**只考虑权重**，至少需要多少张卡才能放下？（向上取整）

In [ ]:
def cards_needed(total_params_b, per_card_gb, bytes_per_param=2):
    # TODO: 总权重字节 = total_params_b*1e9*bytes_per_param；
    #       返回 ceil(总字节 / (per_card_gb*1e9))
    raise NotImplementedError

In [ ]:
# 自测
import math
n = cards_needed(175, 80)                      # GPT-3 175B, 80GB 卡
assert n == math.ceil(175e9*2 / (80e9))        # = ceil(350/80) = 5
print(f'GPT-3 175B 仅权重(fp16) 至少需 {n} 张 80GB 卡放下')
print('（实际还要存梯度+优化器状态+激活，需求是这个的好几倍 —— 见模块 02 的显存账）')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def cards_needed(total_params_b, per_card_gb, bytes_per_param=2):
    total_bytes = total_params_b * 1e9 * bytes_per_param
    return math.ceil(total_bytes / (per_card_gb * 1e9))

---
## 🔧 对照：真实 Megatron-LM 的张量并行长什么样

本课用 numpy `np.split` + 求和模拟的列切/行切，在 Megatron-LM 里是两个真实的层（伪代码，**本环境不跑**）：

```python
# Megatron-LM core（简化）
class ColumnParallelLinear(nn.Module):       # 对应我们的「列切分」
    def forward(self, x):
        # x 已是完整输入（前向 identity，反向 all-reduce 梯度）
        x = copy_to_tensor_model_parallel_region(x)
        return F.linear(x, self.weight)       # self.weight 是 A_i = [h, 4h/t]

class RowParallelLinear(nn.Module):          # 对应我们的「行切分 + all-reduce」
    def forward(self, x):                     # x 是分片的 Y_i
        y = F.linear(x, self.weight)          # 部分和 Y_i @ B_i
        return reduce_from_tensor_model_parallel_region(y)  # ★ all-reduce 求和

# 一个 FFN 块就是 ColumnParallelLinear -> GeLU -> RowParallelLinear
# 前向恰好 1 次 all-reduce（在 RowParallel 里），与我们 worked 3 数的一致！
```

对应关系：`np.split(A, t, axis=1)` ↔ 每卡持有 `ColumnParallelLinear.weight`；`all_reduce_sum(partial)` ↔ `reduce_from_tensor_model_parallel_region`。
我们验证过的「列切→行切→一次 all-reduce」结构，与 Megatron 生产代码**逐行对应**。

### 小结
- **模型并行两条路**：TP 把单层横向劈开（列切→行切），PP 把层序列纵向切段。
- **Megatron FFN 块**：列切分(零通信) → GeLU → 行切分 → **1 次 all-reduce**；注意力按头切，同样 1 次。一个 Transformer 层前向 2 次 all-reduce → TP 通信密集，放 NVLink 内、度数 ≤8。
- **PP 的 bubble**：`(p-1)/(m+p-1)`；micro-batch 越多 bubble 越小（m≥4p）。
- **1F1B**：bubble 与 GPipe 相同，但激活峰值从 m 份降到 ~p 份 —— 这才是它的价值。
- **3D 并行**：`world=dp×tp×pp`，TP 在内、PP 居中、DP 在外，按通信频率匹配互联带宽。

下一站：**模块 04 · Checkpoint 与容错** —— 几百张卡跑几周，挂了怎么办？